<a href="https://colab.research.google.com/github/azcsprof/ASU-CSE475-SS25/blob/Unit-4-ICE-2/Unit_4_ICE_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###  RNN Debugging Tool: ICE Prep

This tool helps you understand how the shapes change during a forward pass through an RNN.

####  Goals:
- Understand what `nn.RNN` returns
- Calculate and reshape the output correctly
- Trace shapes for `h0`, `out`, `flatten`, and `Linear`

Use this to guide your work on `rnn.py`. Try changing `hidden_size` or `sequence_length` and watch the shape math in action.

In [ ]:
# RNN Forward Pass Debug Tool (Prep for ICE: rnn.py)

import torch
import torch.nn as nn

# STEP 1: Set RNN Hyperparameters (Adjust Here)

batch_size = 10
sequence_length = 5
input_feature_size = 512
hidden_size = 1024
num_layers = 2
num_classes = 3

###  What These Hyperparameters Do (and What to Try)

| Hyperparameter        | Role in the Model                                                 | Suggested Values       | Notes                                                       |
|-----------------------|-------------------------------------------------------------------|------------------------|-------------------------------------------------------------|
| `batch_size`          | Number of sequences processed at once                            | 1, 10, 32              | Affects batch parallelism, but not shape logic in this ICE |
| `sequence_length`     | Number of time steps per sequence                                | 3, 5, 8                | Controls how many hidden states are flattened               |
| `input_feature_size`  | Number of features per time step                                 | 128, 256, 512          | Defines input dimensionality to RNN                         |
| `hidden_size`         | Size of the RNN’s memory vector                                  | 256, 512, 1024         | Affects model capacity and final FC input size              |
| `num_layers`          | Number of stacked RNN layers                                     | 1, 2, 3                | Affects shape of `h0`, not shape of `out`                   |
| `num_classes`         | Number of class scores output by the model                       | 3, 5, 10               | Controls final output size of the model                     |

---

 Try changing **one value at a time**, re-running the model cell, and watching the printed shapes.

 Ask yourself:
- What shape changed?
- Why did it change?
- What part of the code caused that change?

This is how you build shape intuition for neural network design.

In [ ]:
# Step 2: Define the RNN model
class RNN(nn.Module):
    def __init__(self):
        super(RNN, self).__init__()
        self.rnn = nn.RNN(
            input_size=input_feature_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(sequence_length * hidden_size, num_classes)

    def forward(self, x):
        print(" Input x shape:", x.shape)  # (batch_size, sequence_length, input_feature_size)

        h0 = torch.zeros(num_layers, x.size(0), hidden_size)
        print(" Initial h0 shape:", h0.shape)  # (num_layers, batch_size, hidden_size)

        out, h_last = self.rnn(x, h0)
        print(" RNN output shape (all time steps):", out.shape)  # (batch_size, sequence_length, hidden_size)

        out_flat = out.reshape(x.size(0), -1)
        print(" Flattened out shape:", out_flat.shape)  # (batch_size, sequence_length * hidden_size)

        out_final = self.fc(out_flat)
        print(" Final output shape:", out_final.shape)  # (batch_size, num_classes)

        return out_final

# Step 3: Run the model
model = RNN()
output = model(x)

# Step 4: FC layer parameters
for name, param in model.named_parameters():
    if name == "fc.weight":
        print(" fc.weight shape:", param.shape)  # (num_classes, sequence_length * hidden_size)

###  Shape Reference Table

| Variable        | Shape                                     | Explanation                                                  |
|-----------------|--------------------------------------------|--------------------------------------------------------------|
| `x`             | `(batch_size, sequence_length, input_dim)` | Input tensor: one sequence per example                      |
| `h0`            | `(num_layers, batch_size, hidden_size)`    | Initial hidden state for each layer                         |
| `out`           | `(batch_size, sequence_length, hidden_size)` | RNN output for all time steps                              |
| `out_flat`      | `(batch_size, sequence_length × hidden_size)` | Flattened output: one vector per sequence                |
| `fc.weight`     | `(num_classes, sequence_length × hidden_size)` | Fully connected projection to class scores               |
| `output`        | `(batch_size, num_classes)`                | Final prediction logits per input sequence                  |